In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm

In [ ]:
# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# Siia salvestuvad loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"


In [2]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')

#### mitu korda iga root esineb igas mustris

In [4]:
query = """
SELECT root_word, verb as pat, count(head_id) as head_cnt 
FROM transactions_verbs_obl_kohakaandes
group by root_word, pat
order by head_cnt desc
"""

source = pd.read_sql_query(query, con)

source

,root_word,pat,head_cnt
0,mina,meeldima,24694
1,mina,tulema,14971
2,mina,olema,14606
3,käsi,saama,13913
4,see,saama,12286
...,...,...,...
1672117,α-aminohappejääk,koosnema,1
1672118,β-glükaan-solubilaa,osalema,1
1672119,ω-3-rasvhape,moodustama,1
1672120,ω-3-rasvhape,suurenema,1


In [6]:
source2 = source[source["head_cnt"]>1]
source2

,root_word,pat,head_cnt
0,mina,meeldima,24694
1,mina,tulema,14971
2,mina,olema,14606
3,käsi,saama,13913
4,see,saama,12286
...,...,...,...
555284,žürii,valima,2
555285,žürii,viibima,2
555286,žüriiliige,jääma,2
555287,žüriiliige,koguma,2


In [13]:
# kuna muidu on roote liiga palju, siis võtta ainult need, millel on head_cnt > 1
roots=list(set(list(source2["root_word"])))
len(roots)

55103

In [14]:
pats = list(set(list(source2["pat"])))
len(pats)

4066

In [17]:
num_cols = len(pats)+2
data = [[r]+[0]*num_cols for r in roots]

In [18]:
df = pd.DataFrame(data, columns = ["root"]+pats+["total_count", "pat_count"])

In [19]:
df

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima,total_count,pat_count
0,suusamägi,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Shanghai,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,mõistava,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Chicaco,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ühinemisprotsess,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55098,pindalatoetus,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55099,finantsnäitaja,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55100,tähenärimine,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55101,emaettevõte,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
for i in tqdm(range(len(source2))):
    root = source2.iloc[i]["root_word"]
    pat = source2.iloc[i]["pat"]
    
    rownr = df.loc[df['root'] == root].index[0]
    df.at[rownr, pat] = source2.iloc[i]["head_cnt"]

100%|██████████████████████████████████| 555289/555289 [54:57<00:00, 168.38it/s]


In [23]:
df['total_count'] = df.sum(axis=1, numeric_only=True)

In [24]:
df

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima,total_count,pat_count
0,suusamägi,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,0
1,Shanghai,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3,0,0,95,0
2,mõistava,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
3,Chicaco,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
4,ühinemisprotsess,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55098,pindalatoetus,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
55099,finantsnäitaja,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
55100,tähenärimine,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
55101,emaettevõte,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,23,0


In [ ]:
#cur.execute("""
#DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_patterns_stats_v1
#""")

In [ ]:
#df.to_sql(name='transactions_verbs_obl_kohakaandes_patterns_stats_v1', con=con)

In [25]:
df.to_csv("transactions_verbs_obl_kohakaandes_patterns_stats_v1.csv", sep=",", index=False, encoding="utf-8")

### matchide count asemel ainult ühed ja loenda mustreid

In [2]:

df = pd.read_csv("transactions_verbs_obl_kohakaandes_patterns_stats_v1.csv",sep=",", encoding="utf-8")

In [4]:
df = df.drop(['total_count'], axis=1)

In [5]:
df

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,jääma,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima,pat_count
0,suusamägi,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Shanghai,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,3,0,0,0
2,mõistava,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Chicaco,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ühinemisprotsess,0,0,0,0,0,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55098,pindalatoetus,0,0,0,0,0,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
55099,finantsnäitaja,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55100,tähenärimine,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
55101,emaettevõte,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
%%time

columns = list(df.columns)[1:-1]

for col in columns:
    df.loc[df[col] > 0, col] = 1

CPU times: user 2.38 s, sys: 1.21 ms, total: 2.38 s
Wall time: 2.38 s


In [7]:
df['pat_count'] = 0

In [8]:
df['pat_count'] = df.sum(axis=1, numeric_only=True)

In [9]:
df

,root,opteerima,korrigeerima,affima,puhkama,eitama,saaxima,saukima,haugatama,trahvima,...,jääma,voolama,pilutama,teisendama,krissuma,töusnuma,käima,ammuma,meilima,pat_count
0,suusamägi,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
1,Shanghai,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,33
2,mõistava,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
3,Chicaco,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,ühinemisprotsess,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55098,pindalatoetus,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
55099,finantsnäitaja,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
55100,tähenärimine,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
55101,emaettevõte,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7


In [10]:
df.to_csv("transactions_verbs_obl_kohakaandes_patterns_stats_v2.csv", sep=",", index=False, encoding="utf-8")

In [5]:
con.close()